In [0]:
# Rescore Harness v4 — Provable Resilience Intelligence
# @ml-engineer | P35-ML-RESCORE-HARNESS-V3-PARITY
# Parameterized scoring pipeline: loads champion model from UC,
# rebuilds features from source tables, scores all active spans,
# writes to _v4_candidate table, and validates parity against v3.
# Branding: Lakelink Fiber (per CEO directive)

import mlflow
import mlflow.pyfunc
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import json
from datetime import datetime

SCHEMA = "cdm_tmforum.provable_resilience_intelligence"
MODEL_NAME = "cdm_tmforum.provable_resilience_intelligence.span_risk_classifier"
MODEL_ALIAS = "champion"  # v7
OUTPUT_TABLE = f"{SCHEMA}.span_risk_predictions_v4_candidate"
BASELINE_TABLE = f"{SCHEMA}.span_risk_predictions"
EXPERIMENT_NAME = "/Users/stephen.hage@databricks.com/provable-resilience-intelligence/rescore_harness_v4"

# v3 baseline KPIs (from P34 session)
V3_BASELINE = {
    "avg_risk_score": 61.55,
    "critical_count": 848,
    "annual_value_m": 48.45
}

# Parity thresholds
PARITY_CLASS_MATCH_THRESHOLD = 0.99  # 99% class match required for parity

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Harness v4 initialized | Model: {MODEL_NAME}@{MODEL_ALIAS}")
print(f"Output → {OUTPUT_TABLE}")
print(f"Baseline ← {BASELINE_TABLE}")

In [0]:
# Load the champion model from Unity Catalog
model_uri = f"models:/{MODEL_NAME}@{MODEL_ALIAS}"
champion_model = mlflow.pyfunc.load_model(model_uri)

# Get model version info
from mlflow import MlflowClient
client = MlflowClient()
version_info = client.get_model_version_by_alias(MODEL_NAME, MODEL_ALIAS)
model_version = version_info.version
model_run_id = version_info.run_id

print(f"Loaded champion model v{model_version}")
print(f"  Run ID: {model_run_id}")

# Get the original run metrics for reference
orig_run = client.get_run(model_run_id)
print(f"  Features: {orig_run.data.params.get('n_features', 'unknown')}")
print(f"  XGBoost: {orig_run.data.params.get('xgboost', 'unknown')}")

In [0]:
# ============================================================
# FEATURE LOADING — Champion v7 expects 35 features
# For PARITY RUN: load from span_risk_inference_log (v3 pins)
# For V4 RESCORE: swap to hazard_span_intersection_v4_candidate
# ============================================================

# The 35 features the champion v7 model expects
FEATURE_COLS = [
    "fiber_count", "wavelength_count", "length_km", "span_age_years",
    "total_alarms", "distinct_alarm_types", "critical_alarm_count",
    "major_alarm_count", "severe_alarm_count", "osnr_degrade_count",
    "ber_high_count", "fiber_cut_count", "los_count",
    "avg_alarm_duration_hrs", "unack_alarm_count", "days_since_last_critical",
    "srlg_count", "srlg_type_count", "srlg_critical_count", "srlg_severe_count",
    "service_count", "spans_in_conduit",
    "wildfire_count", "flood_count", "dig_ticket_count", "weather_count",
    "earthquake_count", "hazard_total_count", "hazard_critical_count",
    "avg_optical_power_dbm", "avg_osnr_db", "avg_ber", "avg_pmd_ps",
    "stddev_osnr", "stddev_ber"
]
assert len(FEATURE_COLS) == 35, f"Expected 35 features, got {len(FEATURE_COLS)}"

# --- MODE SWITCH ---
# Parity mode: load from inference_log (known-good v3 pins)
# V4 mode: TODO — swap to hazard_span_intersection_v4_candidate when BUG-QA-065 resolves
USE_V4_HAZARD = False  # Flip to True when v4 candidate table lands

if USE_V4_HAZARD:
    # V4 mode: load from v4 candidate intersection table
    source_table = f"{SCHEMA}.hazard_span_intersection_v4_candidate"
    print(f"V4 MODE: Loading features from {source_table}")
    feature_src = spark.table(source_table)
else:
    # Parity mode: load from inference_log (v3 pins)
    source_table = f"{SCHEMA}.span_risk_inference_log"
    print(f"PARITY MODE: Loading v3 pin features from {source_table}")
    feature_src = spark.table(source_table)

pdf = feature_src.select(["span_id"] + FEATURE_COLS).toPandas()
pdf = pdf.fillna(0)
for c in FEATURE_COLS:
    pdf[c] = pdf[c].astype(float)

print(f"Feature matrix: {len(pdf)} spans x {len(FEATURE_COLS)} features")
print(f"Source: {source_table}")

In [0]:
# ============================================================
# SCORE WITH CHAMPION MODEL
# ============================================================

predictions = champion_model.predict(pdf[FEATURE_COLS])

# Parse pyfunc output (returns DataFrame with risk_class, risk_label, prob_*)
if isinstance(predictions, pd.DataFrame):
    pred_df = predictions
elif isinstance(predictions, np.ndarray) and len(predictions) > 0 and isinstance(predictions[0], dict):
    pred_df = pd.DataFrame(predictions.tolist())
else:
    pred_df = pd.DataFrame(predictions)

print(f"Prediction columns: {list(pred_df.columns)}")
print(f"Sample: {pred_df.iloc[0].to_dict()}")

In [0]:
# ============================================================
# BUILD OUTPUT TABLE + WRITE TO v4_candidate
# ============================================================

# Assemble output
output_pdf = pdf[['span_id']].copy()

# Handle both pyfunc wrapper output (dict) and raw output
if 'risk_class' in pred_df.columns:
    output_pdf['predicted_risk_class'] = pred_df['risk_class'].values
    output_pdf['predicted_risk_label'] = pred_df['risk_label'].values
    output_pdf['prob_low'] = pred_df['prob_low'].values
    output_pdf['prob_moderate'] = pred_df['prob_moderate'].values
    output_pdf['prob_high'] = pred_df['prob_high'].values
    output_pdf['prob_critical'] = pred_df['prob_critical'].values
else:
    # Fallback: raw class predictions
    output_pdf['predicted_risk_class'] = pred_df.iloc[:, 0].values
    class_map = {0: 'low', 1: 'moderate', 2: 'high', 3: 'critical'}
    output_pdf['predicted_risk_label'] = output_pdf['predicted_risk_class'].map(class_map)

output_pdf['model_version'] = f"v4_harness_parity_v{model_version}"
output_pdf['model_run_id'] = model_run_id
output_pdf['origin'] = 'INFERENCE'

# Convert to Spark and write
output_sdf = spark.createDataFrame(output_pdf)
output_sdf = output_sdf.withColumn("scored_at", F.current_timestamp())

output_sdf.write.mode("overwrite").saveAsTable(OUTPUT_TABLE)

print(f"Wrote {output_sdf.count()} predictions to {OUTPUT_TABLE}")
output_sdf.groupBy("predicted_risk_label").count().orderBy("predicted_risk_label").show()

spark.sql(f"""COMMENT ON TABLE {OUTPUT_TABLE} IS
  'V4 rescore harness candidate predictions. Same champion model (v7 XGBoost) re-scored from source features. Used to prove parity with v3 before v4 hazard data lands. Lakelink Fiber Provable Resilience Intelligence.'""")
print("Table comment set.")

In [0]:
# ============================================================
# PARITY VALIDATION: Compare v4_candidate against v3 baseline
# ============================================================

v4_cand = spark.table(OUTPUT_TABLE).select(
    "span_id",
    F.col("predicted_risk_class").alias("v4_class"),
    F.col("predicted_risk_label").alias("v4_label"),
    F.col("prob_low").alias("v4_prob_low"),
    F.col("prob_moderate").alias("v4_prob_moderate"),
    F.col("prob_high").alias("v4_prob_high"),
    F.col("prob_critical").alias("v4_prob_critical")
)

v3_base = spark.table(BASELINE_TABLE).select(
    "span_id",
    F.col("predicted_risk_class").alias("v3_class"),
    F.col("predicted_risk_label").alias("v3_label"),
    F.col("prob_low").alias("v3_prob_low"),
    F.col("prob_moderate").alias("v3_prob_moderate"),
    F.col("prob_high").alias("v3_prob_high"),
    F.col("prob_critical").alias("v3_prob_critical")
)

joined = v4_cand.join(v3_base, "span_id", "inner")
join_count = joined.count()

print(f"\n{'='*60}")
print(f"PARITY VALIDATION REPORT")
print(f"{'='*60}")
print(f"v4 candidate spans: {v4_cand.count()}")
print(f"v3 baseline spans:  {v3_base.count()}")
print(f"Joined (inner):     {join_count}")

# 1. Exact class match rate
matched = joined.filter(F.col("v4_class") == F.col("v3_class")).count()
class_match_rate = matched / join_count if join_count > 0 else 0
print(f"\n1. Class match rate: {matched}/{join_count} = {class_match_rate:.6f}")
print(f"   Threshold: {PARITY_CLASS_MATCH_THRESHOLD}")
print(f"   Status: {'PASS \u2705' if class_match_rate >= PARITY_CLASS_MATCH_THRESHOLD else 'FAIL \u274c'}")

# 2. Distribution comparison
print("\n2. Label distribution comparison:")
v4_dist = v4_cand.groupBy("v4_label").count().orderBy("v4_label").toPandas()
v3_dist = v3_base.groupBy("v3_label").count().orderBy("v3_label").toPandas()
print(f"   {'Label':<12} {'v3':>7} {'v4':>7} {'Delta':>7}")
for _, row in v4_dist.iterrows():
    v3_c = v3_dist[v3_dist['v3_label'] == row['v4_label']]['count'].values
    v3_c = v3_c[0] if len(v3_c) > 0 else 0
    delta = row['count'] - v3_c
    print(f"   {row['v4_label']:<12} {v3_c:>7} {row['count']:>7} {delta:>+7}")

# 3. Probability drift (mean absolute difference per class)
join_pdf = joined.select(
    (F.abs(F.col("v4_prob_low") - F.col("v3_prob_low"))).alias("d_low"),
    (F.abs(F.col("v4_prob_moderate") - F.col("v3_prob_moderate"))).alias("d_mod"),
    (F.abs(F.col("v4_prob_high") - F.col("v3_prob_high"))).alias("d_high"),
    (F.abs(F.col("v4_prob_critical") - F.col("v3_prob_critical"))).alias("d_crit")
).agg(
    F.avg("d_low").alias("mad_low"),
    F.avg("d_mod").alias("mad_mod"),
    F.avg("d_high").alias("mad_high"),
    F.avg("d_crit").alias("mad_crit"),
    F.max("d_low").alias("max_d_low"),
    F.max("d_mod").alias("max_d_mod"),
    F.max("d_high").alias("max_d_high"),
    F.max("d_crit").alias("max_d_crit")
).collect()[0]

print(f"\n3. Probability drift (mean absolute difference):")
print(f"   {'Class':<12} {'MAD':>10} {'Max':>10}")
for cls, mad_k, max_k in [('low', 'mad_low', 'max_d_low'), ('moderate', 'mad_mod', 'max_d_mod'),
                           ('high', 'mad_high', 'max_d_high'), ('critical', 'mad_crit', 'max_d_crit')]:
    mad_v = join_pdf[mad_k] or 0
    max_v = join_pdf[max_k] or 0
    print(f"   {cls:<12} {mad_v:>10.6f} {max_v:>10.6f}")

# 4. Mismatched spans breakdown
if class_match_rate < 1.0:
    mismatches = joined.filter(F.col("v4_class") != F.col("v3_class"))
    mm_count = mismatches.count()
    print(f"\n4. Mismatches: {mm_count} spans")
    mismatches.groupBy("v3_label", "v4_label").count().orderBy(F.desc("count")).show(10)
else:
    print("\n4. Perfect parity — zero mismatches.")

parity_passed = class_match_rate >= PARITY_CLASS_MATCH_THRESHOLD
print(f"\n{'='*60}")
print(f"PARITY VERDICT: {'PASS \u2705' if parity_passed else 'FAIL \u274c'}")
print(f"Class match: {class_match_rate:.6f} (threshold: {PARITY_CLASS_MATCH_THRESHOLD})")
print(f"{'='*60}")

In [0]:
# ============================================================
# LOG PARITY RUN TO MLFLOW
# ============================================================

with mlflow.start_run(run_name="v4_rescore_harness_parity") as run:
    mlflow.set_tag("project", "provable_resilience_intelligence")
    mlflow.set_tag("brand", "lakelink_fiber")
    mlflow.set_tag("task_code", "P35-ML-RESCORE-HARNESS-V3-PARITY")
    mlflow.set_tag("harness_type", "parity_run")
    mlflow.set_tag("baseline_table", BASELINE_TABLE)
    mlflow.set_tag("output_table", OUTPUT_TABLE)
    mlflow.set_tag("champion_version", str(model_version))
    mlflow.set_tag("champion_run_id", model_run_id)

    mlflow.log_param("n_features", len(FEATURE_COLS))
    mlflow.log_param("model_alias", MODEL_ALIAS)
    mlflow.log_param("baseline_table", BASELINE_TABLE)
    mlflow.log_param("output_table", OUTPUT_TABLE)
    mlflow.log_param("parity_threshold", PARITY_CLASS_MATCH_THRESHOLD)

    # Parity metrics
    mlflow.log_metric("parity_class_match_rate", class_match_rate)
    mlflow.log_metric("parity_passed", 1.0 if parity_passed else 0.0)
    mlflow.log_metric("v4_total_spans", v4_cand.count())
    mlflow.log_metric("v3_total_spans", v3_base.count())
    mlflow.log_metric("joined_spans", join_count)
    mlflow.log_metric("mismatched_spans", join_count - matched)

    # Distribution metrics
    for _, row in v4_dist.iterrows():
        mlflow.log_metric(f"v4_rate_{row['v4_label']}", row['count'] / len(pdf))

    # Probability drift metrics
    for cls, mad_k in [('low', 'mad_low'), ('moderate', 'mad_mod'),
                        ('high', 'mad_high'), ('critical', 'mad_crit')]:
        mlflow.log_metric(f"prob_mad_{cls}", join_pdf[mad_k] or 0)

    # Log feature list
    mlflow.log_text(json.dumps(FEATURE_COLS, indent=2), "feature_columns.json")

    harness_run_id = run.info.run_id
    print(f"MLflow run logged: {harness_run_id}")
    print(f"Parity passed: {parity_passed}")